---
## 8.&nbsp; Challenge 😃
Now that you've learnt how to send and retrieve information, it's your turn to show off your skills. Create multiple tables in SQL for the data you scrapped about cities from Wikipedia. One should just be a table about the cities, the others should be facts about the cities.

| city_id | city |
| --- | --- |
| 1 | Berlin |
| 2 | Hamburg |
| 3 | Munich |

<br>

| City ID | Population | Year Data Retrieved |
|---|---|---|
| 1 | 3,850,809 | 2024 |
| 2 | 1,945,532 | 2024 |
| 3 | 1,512,491 | 2024 |

> **Pro Tip:** Visualise your relational database with pen and paper before you start coding. This can help you to identify any potential problems or inconsistencies in your design, and it can also make the coding process more efficient.

---
## 1.&nbsp; Import libraries 💾
If you haven't already installed sqlalchemy, you will need to. Uncomment the code below, install, and then recomment the code - you only need to install it once.

In [25]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lat_lon_parser import parse    # for decimal coordinates
import os

---
## 2.&nbsp; Creating the cities table with python 🐍
Let's start by creating the original DataFrame, including the repeated data.Webscrap information from Wikipedia for a few cities.

In [26]:
def cities_dataframe(cities):

  city_data = []

  for city in cities:
    url = f"https://www.wikipedia.org/wiki/{city}"
    headers = {'User-Agent': 'Chrome/134.0.0.0'}

    response = requests.get(url, headers=headers)
    city_soup = BeautifulSoup(response.content, 'html.parser')

    # extract the relevant information
    city_latitude = city_soup.find(class_="latitude").get_text()
    city_longitude = city_soup.find(class_="longitude").get_text()
    country = city_soup.find(class_="infobox-data").get_text()

    # keep track of data per city
    city_data.append({"city": city,
                    "country": country,
                    "latitude": parse(city_latitude), # latitude in decimal format
                    "longitude": parse(city_longitude), # longitude in decimal format
                    })

  return pd.DataFrame(city_data)

In [27]:
# call the function
list_of_cities = ["Berlin", "Hamburg", "Munich"]

cities_df = cities_dataframe(list_of_cities)
cities_df

,city,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


In [29]:
cities_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   city       3 non-null      str    
 1   country    3 non-null      str    
 2   latitude   3 non-null      float64
 3   longitude  3 non-null      float64
dtypes: float64(2), str(2)
memory usage: 228.0 bytes


---
## 3.&nbsp; Creating the population table with python 🐍
Webscrap information from Wikipedia.

In [30]:
from datetime import datetime # to get today's date

def populations_dataframe(cities):

    population_data = []

    for city in cities:
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # extract the relevant information
        city_population = city_soup.find(string="Population").find_next("td").get_text()
        city_population_clean = int(city_population.replace(",", ""))
        today = datetime.today().strftime("%d.%m.%Y")
        today = pd.to_datetime(today, format="%d.%m.%Y")

        # keep track of data per city
        population_data.append({"city": city,
                        "population": city_population_clean,
                        "timestamp_population": today
                        })

    return pd.DataFrame(population_data)

In [31]:
# call the populations function
cities = ["Berlin", "Hamburg", "Munich"]

population_df = populations_dataframe(cities)
population_df

,city,population,timestamp_population
0,Berlin,3596999,2026-05-27
1,Hamburg,1973896,2026-05-27
2,Munich,1505005,2026-05-27


---
## 4.&nbsp; Creating the matching cities table with SQL 💻

Ok, now we're ready to store this DataFrame in SQL. Before we can send the information in SQL, we need to make a table that has the same columns and data types to recieve the data. While we are creating a table for cities, we can also create the books table too.

Open MySQL Workbench, open a local connection, and open a new file. Then copy and paste the code from below.

```sql
-- Drop the database if it already exists
DROP DATABASE IF EXISTS wikipedia;

-- Create the database
CREATE DATABASE wikipedia;

-- Use the database
USE wikipedia;

-- Create the 'cities' table
CREATE TABLE cities (
    city_id INT AUTO_INCREMENT, -- Automatically generated ID for each city
    city_name VARCHAR(255) NOT NULL, -- Name of the city
    country VARCHAR(255) NOT NULL,
    latitude FLOAT NOT NULL,
    longitude FLOAT NOT NULL,
    PRIMARY KEY (city_id) -- Primary key to uniquely identify each city
);

-- Create the 'population' table
CREATE TABLE population (
    city_id INT AUTO_INCREMENT, -- Automatically generated ID for each book
    Population INT NOT NULL, -- Population of each city
    timestamp_population DATE NOT NULL,    
	  PRIMARY KEY (city_id, timestamp_population) -- Primary key to uniquely identify each book  
    FOREIGN KEY (city_id) 
		    REFERENCES cities (city_id)
);


## 5.&nbsp; Sending the information from this notebook to sql 📠
To establish a connection with the SQL database, we need to provide the notebook with the necessary information, which we do using the connection string below. You will need to modify only the password variable, which should match the password you set during MySQL Workbench installation.

In [32]:
schema = "wikipedia"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

To send information to our sql databse we use the pandas method `.to_sql()`. The argument `if_exists="append"` says that we don't want to overwrite any existing data, but add on to what is already there.

In [33]:
cities_df.to_sql('cities',
                con=connection_string,
                if_exists='append',
                index=False)

3

Now, have a look at the table `cities` in MySQL Workbench, you should see that the names of the cities have appeared.

## 6.&nbsp; Retrieving information from sql to this notebook 📥
It's not only possible to send information to a SQL database, but also retrieve it too. Using `.read_sql()` in combination with the `connection_string` we can access the required data.

In [34]:
# Step 1: Insert cities_df into MySQL
cities_from_sql = pd.read_sql("cities", con=connection_string)
cities_from_sql

,city_id,city,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575


In [35]:
# Step 2: Retrieve the cities with their auto-generated IDs
cities_in_db = pd.read_sql("SELECT city_id, city FROM cities", con=connection_string)
cities_in_db

,city_id,city
0,1,Berlin
1,2,Hamburg
2,3,Munich


## 7.&nbsp; Preparing and sending the populations table 📚
By extracting the citiess table from our SQL database, we gain access to the unique identifier `city_id` assigned to each city. These `city_id`'s serve as pointers to their corresponding city records, allowing us to seamlessly link the `city_id`'s in the populationss table to their respective citiess in the cities table, thereby completing the populations table.

```sql

-- Create the 'population' table
CREATE TABLE population (
    city_id INT AUTO_INCREMENT, -- Automatically generated ID for each book
    Population INT NOT NULL, -- Population of each city
    timestamp_population DATE NOT NULL,    
	  PRIMARY KEY (city_id, timestamp_population) -- Primary key to uniquely identify each book  
    FOREIGN KEY (city_id) 
		    REFERENCES cities (city_id)
);

In [36]:
# Step 3: Merge population_df with city IDs
merged_population_df = population_df.merge(cities_in_db, on="city", how="left")
merged_population_df

,city,population,timestamp_population,city_id
0,Berlin,3596999,2026-05-27,1
1,Hamburg,1973896,2026-05-27,2
2,Munich,1505005,2026-05-27,3


In [43]:
# Step 4: Prepare final DataFrame
population_df = merged_population_df[['city_id', 'population', 'timestamp_population']]

In [46]:
# Step 5: Insert into the population table
population_df.to_sql('population', 
                     con=connection_string, 
                     if_exists='append', 
                     index=False)


3

In [44]:
population_df

,city_id,population,timestamp_population
0,1,3596999,2026-05-27
1,2,1973896,2026-05-27
2,3,1505005,2026-05-27


Using this same method, we can also perform SQL queries to only bring back certain sections of information instead of the whole DataFrame.

In [45]:
pd.read_sql("""
            SELECT DISTINCT city
            FROM cities
            """,
            con=connection_string)

,city
0,Berlin
1,Hamburg
2,Munich
